# Shallow Diffuse 官方原始环境复现与 governed import 入口

该 Notebook 服务第二条链路: 官方原始环境复现 + governed import。它不把 legacy Stable Diffusion 结果混入 SD3.5 主表, 只生成补充表所需的官方参考记录、环境报告、命令日志、schema、validation report 和 Google Drive 压缩包。

正式逻辑位于 `paper_workflow/colab_utils/shallow_diffuse_official_reference.py`, Notebook 只负责 Colab 冷启动、参数注入、远程执行和打包。


## 运行边界

1. 默认样本数为120, 用于 pilot_paper fixed-FPR=0.01 共同协议下的官方 legacy reference 复现。
2. Notebook 会在 helper 中按 `external_baseline/source_registry.json` 自动补齐 Shallow Diffuse 官方源码快照; 源码快照仍按外部缓存治理, 不进入 git 提交。
3. 该入口默认把 `SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_MODEL_ID` 设为 `Manojb/stable-diffusion-2-1-base`, 该路径是 `stabilityai/stable-diffusion-2-1-base` 不可直接访问时的公开镜像回退。
4. 该入口默认开启 `SLM_WM_SHALLOW_DIFFUSE_PATCH_MODEL_REPOSITORY_LAYOUT=1`, 会移除官方源码中对 `revision='fp16'` 分支的硬编码, 并通过环境变量收敛官方攻击器集合, 避免重型攻击器阻断 pilot_paper reference 复现。
5. 该入口默认开启 `SLM_WM_SHALLOW_DIFFUSE_PREPARE_LOCAL_MODEL_REPOSITORY=1`, 会把公开镜像下载到 `/content/shallow_diffuse_model_repository/stable_diffusion_2_1_base`, 并把本地 `model_index.json` 中的 `CLIPImageProcessor` 补丁为 legacy `transformers==4.23.1` 支持的 `CLIPFeatureExtractor`。
6. 该入口默认开启 `SLM_WM_SHALLOW_DIFFUSE_PREPARE_LEGACY_ENV=1`, 会在 `/content/shallow_diffuse_legacy_env` 中创建隔离 Python 3.9 legacy 环境, 并通过 `SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_PYTHON_EXECUTABLE` 间接调用官方脚本。
7. 默认攻击器集合为 `none`, 用于先证明官方 shallow latent injection、DDIM inversion 与 metric import 链路可运行。若后续需要攻击参考, 可把 `SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_ATTACKER_NAMES` 改为逗号分隔列表, 但重型 VAE / diffusion 攻击会显著增加依赖和运行时间。
8. 该入口默认显式传入官方 README 中的 `--reference_model ViT-g-14` 与 `--reference_model_pretrain laion2b_s12b_b42k`, 避免 OpenCLIP 自动选择不一致。


In [ ]:
import os

from google.colab import drive

drive.mount('/content/drive')
os.environ.setdefault('SLM_WM_PROTOCOL_PROFILE', 'pilot_paper_fixed_fpr_0_01')
os.environ.setdefault('SLM_WM_PROMPT_SET', 'pilot_paper')
os.environ.setdefault('SLM_WM_PROMPT_FILE', 'configs/paper_main_pilot_paper_prompts.txt')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_DRIVE_OUTPUT_DIR', '/content/drive/MyDrive/SLM/pilot_paper_results/shallow_diffuse_official_reference')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_OUTPUT_DIR', 'outputs/shallow_diffuse_official_reference')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_SOURCE_DIR', 'external_baseline/primary/shallow_diffuse/source')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_RUN_NAME', 'shallow_diffuse_official_legacy_reference')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_SAMPLE_COUNT', '120')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_START_INDEX', '0')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_MODEL_ID', 'Manojb/stable-diffusion-2-1-base')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_UPSTREAM_OFFICIAL_MODEL_ID', 'stabilityai/stable-diffusion-2-1-base')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_DATASET', 'Gustavosta/Stable-Diffusion-Prompts')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_IMAGE_LENGTH', '512')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_GUIDANCE_SCALE', '7.5')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_NUM_INFERENCE_STEPS', '50')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_EDIT_TIME_LIST', '0.3')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_W_SEED', '42')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_W_CHANNEL', '3')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_W_PATTERN', 'complex2_ring')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_W_MASK_SHAPE', 'circle')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_W_RADIUS', '10')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_W_MEASUREMENT', 'l1_complex2')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_W_INJECTION', 'complex2')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_REFERENCE_MODEL', 'ViT-g-14')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_REFERENCE_MODEL_PRETRAIN', 'laion2b_s12b_b42k')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_ATTACKER_NAMES', 'none')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_PATCH_MODEL_REPOSITORY_LAYOUT', '1')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_PREPARE_LOCAL_MODEL_REPOSITORY', '1')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_LOCAL_MODEL_REPOSITORY_DIR', '/content/shallow_diffuse_model_repository/stable_diffusion_2_1_base')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_PATCH_MODEL_INDEX_FOR_LEGACY_TRANSFORMERS', '1')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_PYTHON_EXECUTABLE', '')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_PREPARE_LEGACY_ENV', '1')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_LEGACY_ENV_PREFIX', '/content/shallow_diffuse_legacy_env')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_MICROMAMBA_PATH', '/content/bin/micromamba')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_LEGACY_PYTHON_VERSION', '3.9')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_LEGACY_TORCH_SPECS', 'torch==1.13.0+cu117 torchvision==0.14.0+cu117')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_LEGACY_PYTORCH_INDEX_URL', 'https://download.pytorch.org/whl/cu117')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_LEGACY_PACKAGE_SPECS', 'transformers==4.23.1 diffusers==0.11.1 huggingface_hub==0.10.1 datasets==2.6.1 pyarrow<13 fsspec==2022.10.0 numpy==1.24.4 scipy==1.10.1 Pillow==9.5.0 tqdm==4.66.2 scikit-learn==1.3.2 wandb==0.16.6 open_clip_torch==2.7.0 ftfy==6.2.0 regex==2023.12.25 Requests==2.31.0 omegaconf==2.3.0 einops==0.4.1 matplotlib==3.7.5 timm==0.5.4 opencv-python-headless==4.9.0.80')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_RUN_COMMAND', '1')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_REQUIRE_CUDA', '1')
os.environ.setdefault('WANDB_MODE', 'disabled')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_SUMMARY_IMPORT_PATH', '')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_LOG_IMPORT_PATH', '')
os.environ.setdefault('SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_TIMEOUT_SECONDS', '86400')
print({
    'drive_output_dir': os.environ['SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_DRIVE_OUTPUT_DIR'],
    'sample_count': os.environ['SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_SAMPLE_COUNT'],
    'official_model_id': os.environ['SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_MODEL_ID'],
    'edit_time_list': os.environ['SLM_WM_SHALLOW_DIFFUSE_EDIT_TIME_LIST'],
    'attacker_names': os.environ['SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_ATTACKER_NAMES'],
    'reference_model': os.environ['SLM_WM_SHALLOW_DIFFUSE_REFERENCE_MODEL'],
    'reference_model_pretrain': os.environ['SLM_WM_SHALLOW_DIFFUSE_REFERENCE_MODEL_PRETRAIN'],
    'prepare_legacy_env': os.environ['SLM_WM_SHALLOW_DIFFUSE_PREPARE_LEGACY_ENV'],
    'legacy_env_prefix': os.environ['SLM_WM_SHALLOW_DIFFUSE_LEGACY_ENV_PREFIX'],
    'official_python_executable': os.environ['SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_PYTHON_EXECUTABLE'] or 'helper_managed_legacy_python',
    'run_command': os.environ['SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_RUN_COMMAND'],
})


In [ ]:
%pip install -q --upgrade packaging huggingface_hub


In [ ]:
import os
import subprocess
import sys

repository_url = os.environ.get('SLM_WM_REPOSITORY_URL', 'https://github.com/RICHAAARC/SLM-WM.git')
repository_ref = os.environ.get('SLM_WM_REPOSITORY_REF', 'main')
workspace_dir = os.environ.get('SLM_WM_WORKSPACE_DIR', '/content/slm_wm_repository')

if not os.path.exists(workspace_dir):
    subprocess.run(['git', 'clone', repository_url, workspace_dir], check=True)

subprocess.run(['git', 'fetch', '--all'], cwd=workspace_dir, check=True)
subprocess.run(['git', 'checkout', repository_ref], cwd=workspace_dir, check=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)
print({'workspace_dir': workspace_dir, 'repository_ref': repository_ref})


In [ ]:
import os

try:
    from google.colab import userdata
    token_from_secret = userdata.get('HF_TOKEN')
except Exception:
    token_from_secret = None

if token_from_secret and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = token_from_secret

if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])
    print('huggingface_login_ready')
else:
    print('HF_TOKEN_not_found')


In [ ]:
from pathlib import Path

source_dir = Path(os.environ['SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_SOURCE_DIR'])
entrypoint_path = source_dir / 'run_shallow_diffuse_t2i.py'
readme_path = source_dir / 'README.md'
print({
    'source_dir': str(source_dir),
    'source_dir_exists_before_helper': source_dir.exists(),
    'entrypoint_exists_before_helper': entrypoint_path.exists(),
    'readme_exists_before_helper': readme_path.exists(),
    'source_prepare_policy': 'helper_will_clone_from_external_baseline_source_registry_when_missing',
})
if readme_path.exists():
    print(readme_path.read_text(encoding='utf-8')[:4000])


In [ ]:
import torch

device_report = {
    'cuda_available': torch.cuda.is_available(),
    'device_count': torch.cuda.device_count(),
    'device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
}
print(device_report)


In [ ]:
from pathlib import Path

from paper_workflow.colab_utils.shallow_diffuse_official_reference import run_default_shallow_diffuse_official_reference_plan

shallow_diffuse_official_reference_summary = run_default_shallow_diffuse_official_reference_plan(root='.')
legacy_prepare_path = Path('outputs/shallow_diffuse_official_reference/shallow_diffuse_legacy_environment_prepare_result.json')
model_prepare_path = Path('outputs/shallow_diffuse_official_reference/shallow_diffuse_model_repository_prepare_result.json')
source_patch_path = Path('outputs/shallow_diffuse_official_reference/shallow_diffuse_official_source_patch_result.json')
for diagnostic_path in (legacy_prepare_path, model_prepare_path, source_patch_path):
    if diagnostic_path.exists():
        print(diagnostic_path)
        print(diagnostic_path.read_text(encoding='utf-8')[:6000])
shallow_diffuse_official_reference_summary


In [ ]:
import os
import subprocess
from datetime import datetime, timezone

from paper_workflow.colab_utils.shallow_diffuse_official_reference import package_shallow_diffuse_official_reference_outputs


def resolve_short_commit():
    try:
        result = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True, capture_output=True, text=True)
    except Exception:
        return 'git_unknown'
    return result.stdout.strip() or 'git_unknown'


drive_output_dir = os.environ.get('SLM_WM_SHALLOW_DIFFUSE_OFFICIAL_DRIVE_OUTPUT_DIR', '/content/drive/MyDrive/SLM/pilot_paper_results/shallow_diffuse_official_reference')
utc_suffix = datetime.now(timezone.utc).strftime('%Y%m%dt%H%M%sz')
short_commit = resolve_short_commit()
archive_name = f'shallow_diffuse_official_reference_package_{utc_suffix}_{short_commit}.zip'
archive_record = package_shallow_diffuse_official_reference_outputs(root='.', drive_output_dir=drive_output_dir, archive_name=archive_name)
archive_record.to_dict()


In [ ]:
from pathlib import Path

for result_path in sorted(Path('outputs/shallow_diffuse_official_reference').rglob('*')):
    if result_path.is_file() and result_path.suffix.lower() in {'.json', '.jsonl', '.txt'}:
        print(result_path)
        print(result_path.read_text(encoding='utf-8', errors='ignore')[:4000])

assert shallow_diffuse_official_reference_summary['sample_count'] == 5, shallow_diffuse_official_reference_summary
if shallow_diffuse_official_reference_summary['run_decision'] != 'pass':
    print({
        'shallow_diffuse_official_reference_ready': shallow_diffuse_official_reference_summary.get('shallow_diffuse_official_reference_ready'),
        'unsupported_reason': shallow_diffuse_official_reference_summary.get('unsupported_reason'),
        'legacy_environment_ready': shallow_diffuse_official_reference_summary.get('legacy_environment_ready'),
        'official_command_return_code': shallow_diffuse_official_reference_summary.get('official_command_return_code'),
    })
else:
    assert shallow_diffuse_official_reference_summary['reference_import_ready'] is True, shallow_diffuse_official_reference_summary
    assert shallow_diffuse_official_reference_summary['governed_reference_record_count'] > 0, shallow_diffuse_official_reference_summary
